*Imports word and trial level dataframes from pipeline*

*Produces all analyses, modeling results, and figures for final paper*


In [ ]:
# Path and logging
import sys
sys.path.append('..')
from utils import load

# Package imports
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import precision_score, recall_score
from sklearn.base import clone
import seaborn as sns
import matplotlib.pyplot as plt

# Notebook output and debugging
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)
pd.options.display.float_format = '{:.3f}'.format
np.random.seed(42)

# Visual styling
plt.rcParams.update({
    'font.family': 'serif',
    'font.serif': 'Times New Roman',
    'font.size': 10,
    'axes.labelsize': 11,
    'axes.titlesize': 12,
    'legend.fontsize': 9,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
    'axes.linewidth': 0.8,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
})
BLUE = '#2563EB'
ORANGE = '#F97316'
GRAY = "#B5B7BB"
TRAIN_COLOR = '#2563EB'
TEST_COLOR = '#F97316'

In [ ]:
# Create Dataframes for all analysis
# DF_PIPE: cg_df_trials for Korean and English
#   Word level trial results for all parameter combinations and random seeds, for Train/Test/Gap.
# DF_AUC:  cg_df_auc for Korean and English
#   Trial level AUC results for Test vs Gap CSLS metrics

# Set local paths
main_path = ''
SUFFIX = ''

# DF_PIPE
DF_K = load(f'{main_path}/K/cg_df_trials{SUFFIX}.pkl')
DF_K['language'] = 'korean'
DF_E = load(f'{main_path}/E/cg_df_trials{SUFFIX}.pkl')
DF_E['language'] = 'english'
DF_PIPE = pd.concat([DF_K, DF_E])
del DF_K
del DF_E

# DF_AUC
DF_K = load(f'{main_path}/K/cg_df_aucs{SUFFIX}.pkl')
DF_K['language'] = 'korean'
DF_E = load(f'{main_path}/E/cg_df_aucs{SUFFIX}.pkl')
DF_E['language'] = 'english'
DF_AUC = pd.concat([DF_K, DF_E])
del DF_K
del DF_E

---

# **Trial metric distributions and Test vs Gap AUC's**

> **Figure 5: Distribution of train and test precision@1 performance by embedding transformation combinations**

In [ ]:
TITLE_SIZE = 18
LABEL_SIZE = 15
TICK_SIZE = 13
LEGEND_SIZE = 13

YLIMS = (-0.05, 1.05)

BOX_WIDTH = 0.6
LINE_WIDTH = 0.7
FLIER_SIZE = 2

# Precision @1 by Language: Transformation x Dimensionality reduction (d=256)

# Train and Test only (gap words have no translation)
DF_PLOT = DF_PIPE[~DF_PIPE.eval_type.str.contains('gap')].copy()
# Define precision @ 1
DF_PLOT['precision@1 CSLS'] = DF_PLOT['actual_translation_rank_csls'] == 1
# Mean precision @ 1 across all runs
# (model_family/vector_type (n=10), tf_dm (n=4), eval_type (n=2), random seed (n=100))
DF_PLOT = DF_PLOT.groupby(
    ['language', 'model_family', 'vector_type', 'tf_dm', 'eval_type', 'random_seed']) \
        ['precision@1 CSLS'].mean().reset_index()

fig, axes = plt.subplots(2, 1, figsize=(8, 10), constrained_layout=False)
plt.subplots_adjust(
    hspace=0.42,
    top=0.92,
    bottom=0.10,
    left=0.12,
    right=0.97
)

DF_PLOT.eval_type = DF_PLOT.eval_type.str.title()
DF_PLOT['tf_dm_label'] = DF_PLOT['tf_dm'].map({
    'procrustes-alldims': 'Procrustes\n(All dims)',
    'procrustes-pca256': 'Procrustes\n(PCA-256)',
    'unaligned-alldims': 'Unaligned\n(All dims)',
    'unaligned-pca256': 'Unaligned\n(PCA-256)'
})

for idx, (ax, language) in enumerate(zip(axes.flatten(), ['korean', 'english'])):

    df_plot = DF_PLOT[DF_PLOT.language==language].sort_values(
        ['eval_type', 'tf_dm'], ascending=[False, True])

    sns.boxplot(
        data=df_plot,
        x='tf_dm_label',
        y='precision@1 CSLS',
        hue='eval_type',
        palette={'Train': TRAIN_COLOR, 'Test': TEST_COLOR},
        ax=ax,
        linewidth=0.7,
        fliersize=2,
        width=0.6,
        saturation=0.9,
    )

    ax.set_title(f'{language.title()}', loc='center', fontweight='medium', size=TITLE_SIZE)
    ax.set_xlabel('')
    ax.set_ylabel('Precision@1 CSLS', size=LABEL_SIZE)
    ax.set_ylim(YLIMS)
    ax.tick_params(axis='x', labelsize=TICK_SIZE, rotation=0, length=3)
    ax.tick_params(axis='y', labelsize=TICK_SIZE, length=3)
    ax.get_legend().remove()
    ax.yaxis.grid(True, linestyle='-', alpha=0.3, linewidth=0.5)
    ax.set_axisbelow(True)

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(
    handles, labels,
    loc='upper right',
    bbox_to_anchor=(1, 0.94),
    title='Split',
    frameon=False,
    title_fontsize=15,
    fontsize=12
)


plt.show()

> **Table 3: Distribution of train and test precision@1 performance by embedding transformation combinations**

In [ ]:
# Train word overall p@1 for each transformation / dim reduction across all embedding spaces, compare CSLS (figure) to CS (unshown)

df_tbl = DF_PIPE[DF_PIPE.eval_type=='train'].copy()
df_tbl['precision@1 CS'] = df_tbl['actual_translation_rank_cs'] == 1
df_tbl['precision@1 CSLS'] = df_tbl['actual_translation_rank_csls'] == 1
df_tbl.groupby(['language', 'tf_dm'])[['precision@1 CS', 'precision@1 CSLS']].mean()

In [ ]:
# Test word overall p@1 for each transformation / dim reduction across all embedding spaces, compare CSLS (figure) to CS (unshown)

df_tbl = DF_PIPE[DF_PIPE.eval_type=='test'].copy()
df_tbl['precision@1 CS'] = df_tbl['actual_translation_rank_cs'] == 1
df_tbl['precision@1 CSLS'] = df_tbl['actual_translation_rank_csls'] == 1
df_tbl.groupby(['language', 'tf_dm'])[['precision@1 CS', 'precision@1 CSLS']].mean()

> **Figure 2: Distribution of AUCs across all embedding spaces and random seeds for each source language**

In [ ]:
# Violin plot of AUC distributions by language

fig, ax = plt.subplots(figsize=(6, 3))

vp = sns.violinplot(
    data=DF_AUC, x='auc', y='language', order=['korean', 'english'], orient='h', ax=ax,
    palette={'korean': BLUE, 'english': ORANGE},
    inner=None, linewidth=1, cut=0, density_norm='area', alpha=.6)

for i, coll in enumerate(ax.collections):
    coll.set_clip_path(
        plt.Rectangle(
            (-10, i),
            20,
            0.5,
            transform=ax.transData
        )
    )

for i, lang in enumerate(['korean', 'english']):
    d = DF_AUC[DF_AUC.language == lang].auc
    q25, med, q75 = np.percentile(d, [25, 50, 75])
    ax.hlines(i, q25, q75, color='black', linewidth=1.2, zorder=5)
    ax.scatter(med, i, color='white', s=30, zorder=6, edgecolor='black', linewidth=0.9)

ax.axvline(0.5, color='black', linestyle='--', linewidth=1, alpha=.3)
ax.set_xlabel('AUC', fontsize=13)
ax.set_ylabel('')
ax.set_xlim(.2, 1)
ax.set_ylim(-0.5, 1.5)
ax.tick_params(direction='out', length=3, width=0.6, labelsize=11)
ax.set_yticks([0, 1], labels=['Korean', 'English'], fontsize=11)
ax.tick_params(left=False, length=3, width=0.6)

plt.tight_layout()
plt.show()

In [ ]:
# Percent of trials with > .5 AUC, per language

DF_AUC['above_50'] = DF_AUC.auc > .5
DF_AUC.groupby('language').above_50.agg(['mean', 'count'])

In [ ]:
# Spearman correlation coefficient between Korean and English experiment configuration AUCs.
# AUC per experiment configuration calculated as median AUC over 100 random seeds. 

df_corr = DF_AUC.groupby(['language', 'model_family', 'vector_type', 'tf_dm'], as_index=False).auc.median()
df_corr = df_corr.pivot(index=['model_family', 'vector_type', 'tf_dm'], columns=['language'], values=['auc']).reset_index()
spearmanr(df_corr[('auc', 'korean')], df_corr[('auc', 'english')])

> **Table 2: Distribution of test word sampling across 100 random seeds**

In [ ]:
# Distribution of number of test words by number of trials (random seeds, total=100) participated.

num_trials_per_word_src_word = DF_PIPE[DF_PIPE.eval_type.isin(['test'])][['language', 'random_seed', 'eval_type', 'src_word']].drop_duplicates().copy()
num_trials_per_word_src_word = num_trials_per_word_src_word.groupby(['language', 'eval_type', 'src_word'], as_index=False).random_seed.nunique().rename(columns={'random_seed': 'num_random_seeds'})
result = num_trials_per_word_src_word.groupby(['language', 'num_random_seeds', 'eval_type']).src_word.nunique().unstack(['language', 'eval_type']).fillna(0).astype(int)
full_idx = pd.RangeIndex(result.index.min(), result.index.max() + 1, name='num_random_seeds')
result = result.reindex(full_idx, fill_value=0)
result

> **Figure 7: AUCs across all languages and embedding spaces**

In [ ]:
# Plot median AUC across random seeds for each model parameter configuration (PCA = 256)

DF_PLOT = DF_AUC.groupby(
    ['language', 'model_family', 'vector_type', 'tf_dm', 'metric'],
    as_index=False).auc.median().copy()

DF_PLOT.language = DF_PLOT.language.str.title()
DF_PLOT.metric = DF_PLOT.metric.map({
    'neighbor_1_csls': 'NN Similarity (CSLS)'})

tf_dm_labels_map={
    'procrustes-alldims': 'Procrustes\n(All dims)',
    'procrustes-pca256': 'Procrustes\n(PCA-256)',
    'unaligned-alldims': 'Unaligned\n(All dims)',
    'unaligned-pca256': 'Unaligned\n(PCA-256)'}
DF_PLOT.tf_dm = DF_PLOT.tf_dm.map(tf_dm_labels_map)

new_labels_map = {
    'd_exa_24': 'Exaone-3.5-2.4b',
    'd_exa_78': 'Exaone-3.5-7.8b',
    'd_kn_na': 'Kanana-nano',
    'd_kn_21': 'Kanana-1.5-2.1b',
    'd_kn_8': 'Kanana-1.5-8b'
}
DF_PLOT['model_family_new'] = DF_PLOT['model_family'].map(new_labels_map).fillna(DF_PLOT['model_family'])

DF_PLOT.vector_type = DF_PLOT.vector_type.apply(lambda x: '_vector_last' if x=='_vector' else x)
DF_PLOT['model'] = DF_PLOT['model_family_new'] + ' (' + DF_PLOT['vector_type'].str.replace('_vector', '').str.replace('_', '') + ')'

fig, axes = plt.subplots(2, 1, figsize=(8, 8))
plt.subplots_adjust(hspace=0.4, wspace=0.3)

for row, metric in enumerate(DF_PLOT.metric.unique()):

    for col, language in enumerate(['Korean', 'English']):
        ax = axes[col]

        df_plot = DF_PLOT[(DF_PLOT.metric == metric)&(DF_PLOT.language == language)].copy()
        df_plot = df_plot.pivot(index='model', columns='tf_dm', values='auc')

        sns.heatmap(
            df_plot, ax=ax, annot=True, fmt='.3f',
            cmap='RdYlGn', vmin=0, vmax=1.0,
            linewidths=0.5, linecolor='white',
            cbar_kws={'shrink': 0.8}
        )

        ax.set_title(f'{language}', fontweight='medium')
        ax.set_xlabel('')
        ax.set_ylabel('')
        ax.tick_params(axis='both', length=0)

fig.tight_layout()
plt.show()

> **Figure 8: AUCs of models trained using both emotion and non-emotion words across all languages and embedding
spaces.**

In [ ]:
# Set local paths for output with emotion and non-emotion words
main_path = ''
SUFFIX = ''

# DF_AUC_NE
DF_K = load(f'{main_path}/K/cg_df_aucs{SUFFIX}.pkl')
DF_K['language'] = 'korean'
DF_E = load(f'{main_path}/E/cg_df_aucs{SUFFIX}.pkl')
DF_E['language'] = 'english'
DF_AUC_NE = pd.concat([DF_K, DF_E])
del DF_K
del DF_E

# Plot median AUC across random seeds for each model parameter configuration (PCA = 256)

DF_PLOT = DF_AUC_NE.groupby(
    ['language', 'model_family', 'vector_type', 'tf_dm', 'metric'],
    as_index=False).auc.median().copy()

DF_PLOT.language = DF_PLOT.language.str.title()
DF_PLOT.metric = DF_PLOT.metric.map({
    'neighbor_1_csls': 'NN Similarity (CSLS)'})

tf_dm_labels_map={
    'procrustes-alldims': 'Procrustes\n(All dims)',
    'procrustes-pca256': 'Procrustes\n(PCA-256)',
    'unaligned-alldims': 'Unaligned\n(All dims)',
    'unaligned-pca256': 'Unaligned\n(PCA-256)'}
DF_PLOT.tf_dm = DF_PLOT.tf_dm.map(tf_dm_labels_map)

new_labels_map = {
    'd_exa_24': 'Exaone-3.5-2.4b',
    'd_exa_78': 'Exaone-3.5-7.8b',
    'd_kn_na': 'Kanana-nano',
    'd_kn_21': 'Kanana-1.5-2.1b',
    'd_kn_8': 'Kanana-1.5-8b'
}
DF_PLOT['model_family_new'] = DF_PLOT['model_family'].map(new_labels_map).fillna(DF_PLOT['model_family'])

DF_PLOT.vector_type = DF_PLOT.vector_type.apply(lambda x: '_vector_last' if x=='_vector' else x)
DF_PLOT['model'] = DF_PLOT['model_family_new'] + ' (' + DF_PLOT['vector_type'].str.replace('_vector', '').str.replace('_', '') + ')'

fig, axes = plt.subplots(2, 1, figsize=(8, 8))
plt.subplots_adjust(hspace=0.4, wspace=0.3)

for row, metric in enumerate(DF_PLOT.metric.unique()):

    for col, language in enumerate(['Korean', 'English']):
        ax = axes[col]

        df_plot = DF_PLOT[(DF_PLOT.metric == metric)&(DF_PLOT.language == language)].copy()
        df_plot = df_plot.pivot(index='model', columns='tf_dm', values='auc')

        sns.heatmap(
            df_plot, ax=ax, annot=True, fmt='.3f',
            cmap='RdYlGn', vmin=0, vmax=1.0,
            linewidths=0.5, linecolor='white',
            cbar_kws={'shrink': 0.8}
        )

        ax.set_title(f'{language}', fontweight='medium')
        ax.set_xlabel('')
        ax.set_ylabel('')
        ax.tick_params(axis='both', length=0)

fig.tight_layout()
plt.show()

---

# **Logistic Regression Model**

In [ ]:
# Functions to run Logistic Regression

def loo_auc(
        X: np.ndarray,
        y: np.ndarray, 
        clf: LogisticRegression
) -> dict:
    ''' 
    Runs LOO logistic regression predicting for a given classifier configuration. For each n
    observations (words), build model on all other observations, and generates the probability of 
    the left out word being a gap word. Calculates AUC, precision and recall at 0.5 cutoff on
    resulting predictions across all words.

    Args:
        X: array of predictors
        y: array of labels (0=test, 1=gap)
        clf: Classifier configuration

    Returns:
        Dictionary of individual word predictions and model performance metrics.
    '''

    loo = LeaveOneOut()
    
    # Create empty array to save test probabilities during loop
    probs = np.zeros(len(y))  

    # For each left out word (test_idx), build model on all other words (train_idx)
    for train_idx, test_idx in loo.split(X):

        # Scaler fit on training data, applied to training and leave-out test observation.
        scaler = StandardScaler()
        X_train = scaler.fit_transform(X[train_idx])
        X_test = scaler.transform(X[test_idx])

        # Safety for re-using same model each loop iteration
        clf_clone = clone(clf)

        # Fit model
        clf_clone.fit(X_train, y[train_idx])

        # Grab probability for leave-out
        p = clf_clone.predict_proba(X_test)[0, 1]

        # Input probability into probability matrix at test_index position
        probs[test_idx[0]] = float(p)


    # AUC where gap=1. This is the probability that gap word pred prob > test word pred prob.
    auc = roc_auc_score(y, probs)  
        
    # Precision and recall metrics at p > .5
    gap_probs = probs[y == 1]
    n_gap = y.sum()
    retrieved_50 = probs >= 0.5
    gap_retrieved_50 = (gap_probs >= 0.5).sum()
    precision_50 = gap_retrieved_50 / retrieved_50.sum() if retrieved_50.sum() > 0 else 0.0
    recall_50 = gap_retrieved_50 / n_gap
    
    return {
        'probs': probs,
        'auc': auc,
        'gap_retrieved_50': gap_retrieved_50,
        'precision_50': precision_50,
        'recall_50': recall_50}


def run_model(
        df_in: pd.DataFrame,
        language: str,
        c_value: float,
        unaligned_only: bool
) -> dict: 
    ''' 
    Select observations (by language) and predictors (All or Unaligned spaces only) in scope for 
    the model. For each remaining predictor (trial combination), for each word in Test or Gap
    trials, aggregate CSLS metric across all random seeds to median. Resulting data for model input
    is one row per word, where each predictor is a unique combination of trial parameters, and the 
    value is the median CSLS metric across all random seeds for that combination.

    Args:
        language: Korean or English language
        c_value: regularization strength 
        unaligned_only: Whether to include Procrustes transformed spaces or Unaligned spaces only

    Returns:
        Dictionary of individual word predictions and model performance metrics.
    '''    
    
    # Toggle for running all configurations, or only unaligned (no Procrustes transformation)
    if unaligned_only:
        df_model = df_in[df_in.tf_dm.str.contains('unaligned')].copy()
    else:
        df_model = df_in.copy()

    # Remove train words
    df_model = df_model[(df_model.eval_type!='train')&(df_model.language==language)].copy()

    # For each combination of parameters, for each source word, compute median neighbor_1_csls score
    df_model = (
        df_model.groupby(
            ['model_family','vector_type','tf_dm','eval_type','src_word'])
            ['neighbor_1_csls'].median().reset_index()
    )

    # Create predictor columns
    df_model.rename(columns={'neighbor_1_csls': 'predictor_value'}, inplace=True)
    df_model['predictor'] = (
        'pred__' + df_model['model_family']
        + '__' + df_model['vector_type']
        + '__' + df_model['tf_dm']
    )

    # Pivot on eval_type/src_word -> predictors as columns
    df_model = df_model.pivot_table(
        index=['eval_type', 'src_word'],
        columns='predictor',
        values='predictor_value').reset_index(drop=False)
    
    # Positive class: word is a gap word
    df_model['y'] = df_model['eval_type'].eq('gap').astype(int)
    
    # X = predictors values (median CSLS metric across all runs of a given trial combination)
    predictors = [c for c in df_model.columns if c.startswith('pred_')]
    X = df_model[predictors].to_numpy(dtype=float)

    # y = gap (1), test (0)
    y = df_model['y'].values

    # Assert no Nans, equal number of X predictor values and y eval_type assignments
    assert np.isfinite(df_model[predictors].values).all(), f"NaNs present in predictor set"
    assert X.shape[0] == len(y), f"Different number of observations in X and y inputs"

    # L1 logistic regression with weight class balancing
    model = LogisticRegression(
                penalty='l1',
                solver='liblinear',
                C=c_value,
                class_weight='balanced',
                random_state=42,
                max_iter=2000
            )

    # Run LOO model
    model_output = loo_auc(X=X, y=y, clf=model)

    # Assign probabilities to original df with eval_type for plotting
    df_model['prob'] = model_output['probs']

    # Save final dataframe to model_output dictionary
    model_output['df_model'] = df_model
    
    return model_output

In [ ]:
# Run models for Korean and English across regularization strengths.

reg_strengths = [.05, .1, .2, .5 , 1, 2, 5, 10]

l_all = []
l_unaligned = []
for c in reg_strengths:
    
    # Run regularization sweep using all predictors (unaligned_only=False)
    mk = run_model(DF_PIPE, 'korean', c, unaligned_only=False)
    me = run_model(DF_PIPE, 'english', c, unaligned_only=False)
    l_all.append([c, mk['auc'], me['auc'], mk['gap_retrieved_50'], me['gap_retrieved_50']])
    
    # Run regularization sweep using only unaligned predictors (unaligned_only=True)
    mk = run_model(DF_PIPE, 'korean', c, unaligned_only=True)
    me = run_model(DF_PIPE, 'english', c, unaligned_only=True)
    l_unaligned.append([c, mk['auc'], me['auc'], mk['gap_retrieved_50'], me['gap_retrieved_50']])

> **Figure 10: Logistic regression results by L1 regularization parameter**

In [ ]:
def plot_lr_results(model_output: list, ax, title: str):

    DF_PLOT = pd.DataFrame([i[:5] for i in model_output], columns=[
        'c', 'auc_k', 'auc_e', 'gap_retrieved_50_k', 'gap_retrieved_50_e'
    ]).set_index('c')

    annot = DF_PLOT.copy().astype(object)
    for col in ['auc_k', 'auc_e']:
        annot[col] = DF_PLOT[col].map(lambda x: f'{x:.2f}')
    for col in ['gap_retrieved_50_k', 'gap_retrieved_50_e']:
        annot[col] = DF_PLOT[col].map(lambda x: f'{int(x)}')

    nice_labels = {
        'auc_k': 'AUC\n(Korean)',
        'auc_e': 'AUC\n(English)',
        'gap_retrieved_50_k': 'Gap words\nretrieved at p>.5\n(Korean)',
        'gap_retrieved_50_e': 'Gap words\nretrieved at p>.5\n(English)'
    }

    sns.heatmap(DF_PLOT, annot=annot, fmt='', cmap=['white'], linewidths=0.8,
                linecolor='black', cbar=False, ax=ax,
                annot_kws={'fontsize': 11, 'color': 'black'})

    ax.xaxis.tick_top()
    ax.xaxis.set_label_position('top')
    ax.set_xticklabels([nice_labels[c] for c in DF_PLOT.columns], fontsize=9, ha='center')
    ax.set_yticklabels([f'{v:.2f}' for v in DF_PLOT.index], rotation=0, fontsize=11)

    ax.set_title(title, fontsize=11, fontweight='bold', pad=10)
    ax.set_ylabel('C', fontsize=11, fontweight='bold')
    ax.set_xlabel('')

fig, axes = plt.subplots(1, 2, figsize=(12, 3))
plot_lr_results(l_all, axes[0], '(a) All embedding spaces')
plot_lr_results(l_unaligned, axes[1], '(b) Unaligned embedding spaces')

> **Figure 3: Logistic classifier outputs in Korean and English across different predictor sets**

In [ ]:
def plot_lr_pair(model_output_k, model_output_e, savepath=None):
    fig, axes = plt.subplots(
        1, 2,
        figsize=(12, 3.5),
        sharex=True,
        sharey=True
    )
    fig.subplots_adjust(wspace=0.3)

    y_max = 0

    for ax, (lang, model_output, color) in zip(
        axes,
        [('Korean', model_output_k, BLUE), ('English', model_output_e, ORANGE)]
    ):
        df = model_output['df_model']
        x_gap = df.loc[df.eval_type == 'gap', 'prob'].values
        x_test = df.loc[df.eval_type == 'test', 'prob'].values

        sns.histplot(x_gap, bins=30, stat='density', alpha=0.25,
                     color=color, edgecolor='none', ax=ax)
        sns.histplot(x_test, bins=30, stat='density', alpha=0.2,
                     color=GRAY, edgecolor='none', ax=ax)
        sns.kdeplot(x_gap, linewidth=1.8,
                    label=f'Gap words (n={len(x_gap)})', color=color, ax=ax)
        sns.kdeplot(x_test, linewidth=1.8,
                    label=f'Non-gap words (n={len(x_test)})', color=GRAY, ax=ax)

        ax.axvline(0.5, color='#374151', linestyle=':', linewidth=1, alpha=0.8, zorder=0)
        ax.set_xlim(0, 1)
        ax.set_xlabel('Predicted probability of gap word')
        ax.set_ylabel('Density')
        ax.set_title(f'{lang} (AUC = {model_output["auc"]:.2f})', fontweight='medium', pad=10)
        ax.yaxis.grid(True, linestyle='-', alpha=0.3, linewidth=0.5)
        ax.set_axisbelow(True)
        ax.tick_params(direction='out', length=3, width=0.6, labelsize=11)
        ax.legend(frameon=False, loc='upper left', fontsize=11)

        y_max = max(y_max, ax.get_ylim()[1])

    for ax in axes:
        ax.set_ylim(0, y_max)

    sns.despine()
    fig.tight_layout()

    if savepath:
        fig.savefig(savepath, dpi=300)

    return fig, axes

m_k_all = run_model(DF_PIPE, 'korean', 0.1, unaligned_only=False)
m_e_all = run_model(DF_PIPE, 'english', 0.1, unaligned_only=False)
m_k_un  = run_model(DF_PIPE, 'korean', 0.1, unaligned_only=True)
m_e_un  = run_model(DF_PIPE, 'english', 0.1, unaligned_only=True)

fig1, axes1 = plot_lr_pair(
    m_k_all, m_e_all,
    savepath="lr_all_embeddings.pdf"
)

fig2, axes2 = plot_lr_pair(
    m_k_un, m_e_un,
    savepath="lr_unaligned_embeddings.pdf"
)

plt.show()

> **Figure 4: Precision and recall by different probabilities as thresholds.**

In [ ]:
def precision_recall_by_threshold(df_model, thresholds):
    y_true = (df_model['eval_type'] == 'gap').astype(int).values
    probs = df_model['prob'].values

    rows = []
    for p in thresholds:
        y_pred = (probs >= p).astype(int)

        precision = precision_score(y_true, y_pred, zero_division=0)
        recall = recall_score(y_true, y_pred, zero_division=0)

        rows.append({
            'p_threshold': p,
            'precision': precision,
            'recall': recall,
            'predicted_positive': y_pred.sum()
        })

    return pd.DataFrame(rows)

def plot_precision_recall_side_by_side(korean_pr, english_pr):
    
    fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(14, 5), sharey=True )

    for ax, pr_df, title in zip(axes, [korean_pr, english_pr], ['Korean', 'English']): 
        ax.plot(pr_df['p_threshold'], pr_df['precision'], label='Precision')
        ax.plot(pr_df['p_threshold'], pr_df['recall'], label='Recall')

        ax.axvline(0.5, linestyle=':', linewidth=1.5, color='black')
        ax.set_xlim(0.0, 1.0)
        ax.set_title(title, fontsize=11)
        ax.set_xlabel('Probability threshold', fontsize=11)
        ax.tick_params(axis='both', labelsize=11)
        ax.grid(True)
        ax.legend(fontsize=11)

    axes[0].set_ylabel('Score', fontsize=11)

    plt.tight_layout()
    plt.show()

thresholds = np.arange(0.1, 1.0, 0.05)
c = 0.1

mk = run_model(DF_PIPE, 'korean', c, unaligned_only=True)
korean_pr = precision_recall_by_threshold(mk['df_model'], thresholds)
korean_pr['language'] = 'korean'

me = run_model(DF_PIPE, 'english', c, unaligned_only=True)
english_pr = precision_recall_by_threshold(me['df_model'], thresholds)
english_pr['language'] = 'english'

pr_table = pd.concat([korean_pr, english_pr], ignore_index=True)

plot_precision_recall_side_by_side(korean_pr, english_pr)

> **other**

In [ ]:
# List of gap words and probability of gap word based on specific model run (English)
model_output_e = run_model(DF_PIPE, 'english', .1, unaligned_only=True)

k = model_output_e['df_model'][['eval_type', 'src_word', 'prob']]
k = k[k.eval_type=='gap']
k.sort_values('prob')

In [ ]:
# List of gap words and probability of gap word based on specific model run (Korean)
model_output_k = run_model(DF_PIPE, 'korean', .1, unaligned_only=True)

k = model_output_k['df_model'][['eval_type', 'src_word', 'prob']]
k = k[k.eval_type=='gap']
k.sort_values('prob')

------

# PCA sweep

In [ ]:
# Create dataframe combining all PCA sweep runs (16, 32, 64, 128, 256)

# Set local paths
main_path = ''
folder_path = ''

dim_sweep = [16, 32, 64, 128, 256]

def load_one(df_type, n_dims) -> pd.DataFrame:
    """Load Korean + English for a given file kind ('trials' or 'aucs') and suffix."""
    dfs = []
    for lang, code in [('korean', 'K'), ('english', 'E')]:
        df = load(f'{main_path}/{code}/{folder_path}/cg_df_{df_type}_FINAL_{str(n_dims)}.pkl')
        df['language'] = lang
        dfs.append(df)
    df = pd.concat(dfs)
    df['tf_dm'] = df['tf_dm'].str.replace('256', f'_{str(n_dims)}')
    return df

DF_PIPE_PCA = pd.concat([load_one('trials', d) for d in dim_sweep])
DF_AUC_PCA = pd.concat([load_one('aucs', d) for d in dim_sweep])
DF_PIPE_PCA = pd.concat([DF_PIPE_PCA, DF_PIPE[~DF_PIPE.tf_dm.str.contains('pca')]])
DF_AUC_PCA = pd.concat([DF_AUC_PCA, DF_AUC[~DF_AUC.tf_dm.str.contains('pca')]])

> **Figure 5: Distribution of train and test precision@1 performance by different PCA dimensionalities and alignment (all dimensions)**

In [ ]:
TITLE_SIZE = 18
LABEL_SIZE = 15
TICK_SIZE = 13
LEGEND_SIZE = 13

YLIMS = (-0.05, 1.05)

BOX_WIDTH = 0.6
LINE_WIDTH = 0.7
FLIER_SIZE = 2

# Train, Test results only
DF_PLOT = DF_PIPE_PCA[~DF_PIPE_PCA.eval_type.str.contains('gap')].copy()
# Define precision @ 1
DF_PLOT['precision@1 CSLS'] = DF_PLOT['actual_translation_rank_csls'] == 1
# Mean precision @ 1 across all runs (model family / vector_type (n=10), embedding transformation (n=4), eval type (train,test n=2), random seed (n=100))
DF_PLOT = DF_PLOT.groupby(['language', 'model_family', 'vector_type', 'tf_dm', 'eval_type', 'random_seed'])['precision@1 CSLS'].mean().reset_index()

fig, axes = plt.subplots(2, 1, figsize=(8, 10), constrained_layout=False)
plt.subplots_adjust(
    hspace=0.42,
    top=0.92,
    bottom=0.10,
    left=0.12,
    right=0.97
)

tf_label_map = {
    'procrustes-alldims': 'Procrustes\n(All dims)',
    'procrustes-pca_256': 'Procrustes\n(PCA-256)',
    'procrustes-pca_128': 'Procrustes\n(PCA-128)',
    'procrustes-pca_64': 'Procrustes\n(PCA-64)',
    'procrustes-pca_32': 'Procrustes\n(PCA-32)',
    'procrustes-pca_16': 'Procrustes\n(PCA-16)',
    'unaligned-alldims': 'Unaligned\n(All dims)',
    'unaligned-pca_256': 'Unaligned\n(PCA-256)',
    'unaligned-pca_128': 'Unaligned\n(PCA-128)',
    'unaligned-pca_64': 'Unaligned\n(PCA-64)',
    'unaligned-pca_32': 'Unaligned\n(PCA-32)',
    'unaligned-pca_16': 'Unaligned\n(PCA-16)',
}

DF_PLOT.eval_type = DF_PLOT.eval_type.str.title()
DF_PLOT['tf_dm_label'] = DF_PLOT['tf_dm'].map(tf_label_map)
DF_PLOT.sort_values('tf_dm', inplace=True)

tf_order = [
    'Procrustes\n(All dims)',
    'Procrustes\n(PCA-256)',
    'Procrustes\n(PCA-128)',
    'Procrustes\n(PCA-64)',
    'Procrustes\n(PCA-32)',
    'Procrustes\n(PCA-16)',
    'Unaligned\n(All dims)',
    'Unaligned\n(PCA-256)',
    'Unaligned\n(PCA-128)',
    'Unaligned\n(PCA-64)',
    'Unaligned\n(PCA-32)',
    'Unaligned\n(PCA-16)'
    ]

for idx, (ax, language) in enumerate(zip(axes.flatten(), ['korean', 'english'])):

    df_plot = DF_PLOT[DF_PLOT.language==language].sort_values(['eval_type', 'tf_dm'], ascending=[False, True])

    sns.boxplot(
        data=df_plot,
        x='tf_dm_label',
        y='precision@1 CSLS',
        hue='eval_type',
        palette={'Train': TRAIN_COLOR, 'Test': TEST_COLOR},
        ax=ax,
        order = tf_order,
        linewidth=0.7,
        fliersize=2,
        width=0.6,
        saturation=0.9,
    )

    ax.set_title(f'{language.title()}', loc='center', fontweight='medium', size=TITLE_SIZE)
    ax.set_xlabel('')
    ax.set_ylabel('Precision@1 CSLS', size=LABEL_SIZE)
    ax.set_ylim(YLIMS)
    ax.tick_params(axis='x', labelsize=TICK_SIZE, rotation=45, length=3)
    ax.tick_params(axis='y', labelsize=TICK_SIZE, length=3)
    ax.get_legend().remove()
    ax.yaxis.grid(True, linestyle='-', alpha=0.3, linewidth=0.5)
    ax.set_axisbelow(True)

handles, labels = axes[0].get_legend_handles_labels()


fig.legend(
    handles, labels,
    loc='upper right',
    bbox_to_anchor=(1, 0.94),
    title='Split',
    frameon=False,
    title_fontsize=15,
    fontsize=12
)

plt.show()

> **Figure 9: Gap vs Non-gap Test word AUCs across all languages and embedding spaces with varying PCA dimensionalities.**

In [ ]:
DF_AUC_AGG = DF_AUC_PCA.groupby(['language', 'model_family', 'vector_type', 'tf_dm', 'metric'], as_index=False).auc.median()

# Plot median aucs across random seeds for each model family + vector type, embedding transformation, metric combination
DF_PLOT = DF_AUC_AGG.copy()

# vertically stacked plots
fig, axes = plt.subplots(2, 1, figsize=(8, 10))
plt.subplots_adjust(hspace=0.45)

DF_PLOT.language = DF_PLOT.language.str.title()
DF_PLOT.metric = DF_PLOT.metric.map({
    'neighbor_1_csls': 'NN Similarity (CSLS)'})

tf_label_map = {
    'procrustes-alldims': 'Procrustes\n(All dims)',
    'procrustes-pca_256': 'Procrustes\n(PCA-256)',
    'procrustes-pca_128': 'Procrustes\n(PCA-128)',
    'procrustes-pca_64': 'Procrustes\n(PCA-64)',
    'procrustes-pca_32': 'Procrustes\n(PCA-32)',
    'procrustes-pca_16': 'Procrustes\n(PCA-16)',
    'unaligned-alldims': 'Unaligned\n(All dims)',
    'unaligned-pca_256': 'Unaligned\n(PCA-256)',
    'unaligned-pca_128': 'Unaligned\n(PCA-128)',
    'unaligned-pca_64': 'Unaligned\n(PCA-64)',
    'unaligned-pca_32': 'Unaligned\n(PCA-32)',
    'unaligned-pca_16': 'Unaligned\n(PCA-16)',
}

DF_PLOT['tf_dm_label'] = DF_PLOT['tf_dm'].map(tf_label_map)

new_labels_map = {
    'd_exa_24': 'Exaone-3.5-2.4b',
    'd_exa_78': 'Exaone-3.5-7.8b',
    'd_kn_na': 'Kanana-nano',
    'd_kn_21': 'Kanana-1.5-2.1b',
    'd_kn_8': 'Kanana-1.5-8b'
}
DF_PLOT['model_family_new'] = DF_PLOT['model_family'].map(new_labels_map).fillna(DF_PLOT['model_family'])

DF_PLOT.vector_type = DF_PLOT.vector_type.apply(lambda x: '_vector_last' if x=='_vector' else x)
DF_PLOT['model'] = DF_PLOT['model_family_new'] + ' (' + DF_PLOT['vector_type'].str.replace('_vector', '').str.replace('_', '') + ')'

tf_order = [
    'Procrustes\n(All dims)',
    'Procrustes\n(PCA-256)',
    'Procrustes\n(PCA-128)',
    'Procrustes\n(PCA-64)',
    'Procrustes\n(PCA-32)',
    'Procrustes\n(PCA-16)',
    'Unaligned\n(All dims)',
    'Unaligned\n(PCA-256)',
    'Unaligned\n(PCA-128)',
    'Unaligned\n(PCA-64)',
    'Unaligned\n(PCA-32)',
    'Unaligned\n(PCA-16)'
    ]

# For each metric
for row, metric in enumerate(DF_PLOT.metric.unique()):

    # For each language
    for col, language in enumerate(['Korean', 'English']):
        ax = axes[col]

        # Filter to metric and language
        df_plot = DF_PLOT[(DF_PLOT.metric == metric)&(DF_PLOT.language == language)].copy()

        df_plot = df_plot.pivot(index='model', columns='tf_dm_label', values='auc')
        df_plot = df_plot.reindex(columns=tf_order)

        sns.heatmap(
            df_plot, ax=ax, annot=True, fmt='.3f',
            cmap='RdYlGn', vmin=0, vmax=1.0,
            linewidths=0.5, linecolor='white',
            cbar_kws={'shrink': 0.8}
        )

        ax.set_title(f'{language}', fontweight='medium')
        ax.set_xlabel('')
        ax.set_ylabel('')
        ax.tick_params(axis='x', rotation=45, length=0)
        ax.tick_params(axis='y', length=0)

fig.tight_layout()
plt.show()

In [ ]:
# Run models for Korean and English across regularization strengths.

reg_strengths = [.05, .1, .2, .5 , 1, 2, 5, 10]

df_ = DF_PIPE_PCA[DF_PIPE_PCA.tf_dm.isin(
    ['unaligned-alldims', 'unaligned-pca_256',
     'procrustes-alldims', 'procrustes-pca_256'])]

l_all = []
l_unaligned = []
for c in reg_strengths:
    
    # Run regularization sweep using all predictors (unaligned_only=False)
    mk = run_model(df_,'korean', c, unaligned_only=False)
    me = run_model(df_,'english', c, unaligned_only=False)
    l_all.append([c, mk['auc'], me['auc'], mk['gap_retrieved_50'], me['gap_retrieved_50']])
    
    # Run regularization sweep using only unaligned predictors (unaligned_only=True)
    mk = run_model(df_,'korean', c, unaligned_only=True)
    me = run_model(df_,'english', c, unaligned_only=True)
    l_unaligned.append([c, mk['auc'], me['auc'], mk['gap_retrieved_50'], me['gap_retrieved_50']])

fig, axes = plt.subplots(1, 2, figsize=(12, 3))
plot_lr_results(l_all, axes[0], '(a) All embedding spaces')
plot_lr_results(l_unaligned, axes[1], '(b) Unaligned embedding spaces')    

In [ ]:
# Run models for Korean and English across regularization strengths (PCA=64 for Procrustes).

reg_strengths = [.05, .1, .2, .5 , 1, 2, 5, 10]

df_ = DF_PIPE_PCA[DF_PIPE_PCA.tf_dm.isin(
    ['unaligned-alldims', 'unaligned-pca_256',
     'procrustes-alldims', 'procrustes-pca_64'])]

l_all = []
l_unaligned = []
for c in reg_strengths:
    
    # Run regularization sweep using all predictors (unaligned_only=False)
    mk = run_model(df_,'korean', c, unaligned_only=False)
    me = run_model(df_,'english', c, unaligned_only=False)
    l_all.append([c, mk['auc'], me['auc'], mk['gap_retrieved_50'], me['gap_retrieved_50']])
    
    # Run regularization sweep using only unaligned predictors (unaligned_only=True)
    mk = run_model(df_,'korean', c, unaligned_only=True)
    me = run_model(df_,'english', c, unaligned_only=True)
    l_unaligned.append([c, mk['auc'], me['auc'], mk['gap_retrieved_50'], me['gap_retrieved_50']])

fig, axes = plt.subplots(1, 2, figsize=(12, 3))
plot_lr_results(l_all, axes[0], '(a) All embedding spaces')
plot_lr_results(l_unaligned, axes[1], '(b) Unaligned embedding spaces')    